# Soft Energy-Shell Regularisation — Geodesic Compliance Experiment

## Motivation

The LayerNorm ablation (§3.7 of the companion note) showed that:

1. **Hard** energy-shell projection gives the best directional compliance
   (cosine 0.96–0.97 at layers 2–4) but fails to train (PPL 400).
2. LayerNorm and RMSNorm give identical compliance — normalisation is not
   the primary obstacle to magnitude compliance.
3. The dominant failure mode is the **Christoffel-symbol mismatch** between
   the idealised geodesic model and the full integrator dynamics.

This notebook tests whether a **soft** energy-shell regularisation —
a differentiable penalty added to the training loss — can improve
geodesic magnitude compliance while maintaining trainability.

## Design

The soft energy-shell loss penalises deviations from the damping curve:

$$\mathcal{L}_{\text{shell}} = \frac{\lambda}{L} \sum_{\ell=1}^{L}
  \left( E_\ell - E_0 \, e^{-\gamma \ell} \right)^2$$

where $E_\ell = T_\ell + V_\ell$ is the total energy at layer $\ell$,
$T_\ell = \frac{1}{2} m \|v_\ell\|^2$ is the kinetic energy,
$V_\ell = V_\theta(\xi_\ell, h_\ell)$ is the potential energy,
$E_0 = E_1$ (energy after the first integration step), and $\gamma$ is
the damping coefficient.

The total training objective is:

$$\mathcal{L} = \mathcal{L}_{\text{CE}} + \mathcal{L}_{\text{shell}}$$

## Sweep

| Run | $\lambda$ | LayerNorm | Description |
|-----|-----------|-----------|-------------|
| Baseline | 0 | Yes | Standard LayerNorm, no shell loss |
| Shell-0.01 | 0.01 | Yes | Gentle shell regularisation |
| Shell-0.1 | 0.1 | Yes | Moderate shell regularisation |
| Shell-1.0 | 1.0 | Yes | Strong shell regularisation |
| Shell-0.1-noLN | 0.1 | No | Shell replaces LayerNorm entirely |

In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────────────
import subprocess, sys, os, gc, math, json, time
from pathlib import Path
from dataclasses import asdict, fields as dc_fields

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    run('pip install -q transformers huggingface_hub pyarrow scipy scikit-learn')
    if not os.path.isdir('semsimula-paper'):
        run('git clone --depth 1 https://github.com/dimitarpg13/semsimula-paper.git')
    REPO = 'semsimula-paper'
else:
    REPO = os.environ.get('SEMSIMULA_PAPER', '.')

ARCH_DIR = os.path.join(REPO, 'notebooks', 'conservative_arch')
for p in [
    ARCH_DIR,
    os.path.join(ARCH_DIR, 'multixi'),
    os.path.join(ARCH_DIR, 'parf'),
    os.path.join(ARCH_DIR, 'energetic_minima'),
    os.path.join(ARCH_DIR, 'sarf_mass_variant'),
    os.path.join(ARCH_DIR, 'scaleup'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else ('mps' if hasattr(torch.backends, 'mps')
          and torch.backends.mps.is_available() else 'cpu')
)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for autograd.grad stability')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_soft_energy_shell')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_soft_energy_shell'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_CKPTS = DRIVE_ROOT / 'checkpoints'
DRIVE_CKPTS.mkdir(parents=True, exist_ok=True)
print(f'Drive root   : {DRIVE_ROOT}')
print(f'Results dir  : {DRIVE_RESULTS}')
print(f'Checkpoints  : {DRIVE_CKPTS}')

In [ ]:
# ── Cell 2: Load data (lightweight, OOM-safe) ────────────────────
from data_module import get_batch, _download_hf_parquet, _resolve_tinystories_shard, _gpt2_tokenize
import pyarrow.parquet as pq

SCRIPTS_DIR = os.path.join(ARCH_DIR, 'scaleup')
LOGFREQ_PATH = os.path.join(SCRIPTS_DIR, 'results', 'logfreq_surprisal_tinystories.npy')

if not os.path.exists(LOGFREQ_PATH):
    print('Computing logfreq surprisal (one-time, ~2 min)...')
    os.makedirs(os.path.dirname(LOGFREQ_PATH), exist_ok=True)
    subprocess.run(
        [sys.executable, os.path.join(SCRIPTS_DIR, 'compute_unigram_frequencies_tinystories.py')],
        cwd=SCRIPTS_DIR, check=True,
    )
    print('Done.')

DATA_DIR = os.path.join(ARCH_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

MAX_TRAIN_TOKENS = 6_000_000
MAX_TRAIN_STORIES = 15_000

train_cache = os.path.join(DATA_DIR, 'tinystories_train_capped.npy')
if os.path.exists(train_cache):
    train_ids = np.load(train_cache)
    print(f'Loaded cached train tokens: {len(train_ids):,}')
else:
    print('Loading training data (capped, OOM-safe)...')
    train_fname = _resolve_tinystories_shard("data/train-00000-of-00004")
    tp = _download_hf_parquet("roneneldan/TinyStories", train_fname,
                              "tinystories_train_00000.parquet")
    all_texts = pq.read_table(tp, columns=["text"])["text"].to_pylist()
    n_use = min(len(all_texts), MAX_TRAIN_STORIES)
    print(f'  Tokenising {n_use} stories...')
    chunks = []
    CHUNK = 3000
    for i in range(0, n_use, CHUNK):
        batch_texts = all_texts[i:i+CHUNK]
        chunks.append(_gpt2_tokenize("\n\n".join(batch_texts)))
        del batch_texts
        if sum(len(c) for c in chunks) >= MAX_TRAIN_TOKENS:
            break
    del all_texts
    train_ids = np.concatenate(chunks)[:MAX_TRAIN_TOKENS]
    del chunks
    np.save(train_cache, train_ids)
    print(f'  Cached {len(train_ids):,} train tokens')

gc.collect()

val_cache = os.path.join(DATA_DIR, 'tinystories_val_only.npy')
if os.path.exists(val_cache):
    val_ids = np.load(val_cache)
    print(f'Loaded cached val tokens: {len(val_ids):,}')
else:
    val_fname = _resolve_tinystories_shard("data/validation-00000-of-00001")
    vp = _download_hf_parquet("roneneldan/TinyStories", val_fname,
                              "tinystories_val.parquet")
    val_texts = pq.read_table(vp, columns=["text"])["text"].to_pylist()
    n_stories = min(len(val_texts), 2000)
    val_ids = _gpt2_tokenize("\n\n".join(val_texts[:n_stories]))
    del val_texts
    np.save(val_cache, val_ids)
    print(f'  Cached {len(val_ids):,} val tokens')

gc.collect()
print(f'Train tokens: {len(train_ids):,}  Val tokens: {len(val_ids):,}')

rng = np.random.default_rng(42)

In [ ]:
# ── Cell 3: Model + energy-shell integration patch ─────────────────
from model_multixi import (
    ScalarPotentialLMSARFMassLNMultiXi,
    SPLMSARFMassLNMultiXiConfig,
)
import types


def integrate_with_energy(
    self, x, emb,
    return_trajectory=False, return_xi_trajectory=False,
):
    """Patched integrate() that also returns per-layer energies.

    Returns (h_L, traj_h, traj_xi, energies)
    where energies = list of (T_ell, V_ell) tuples, length L.
    T and V are mean-reduced scalars (differentiable).
    """
    cfg = self.cfg
    h = self._project(emb) if cfg.ln_after_step else emb
    v = torch.zeros_like(h)
    gamma, dt = self.gamma, cfg.dt

    m = self.compute_mass(x, emb)
    m_b = m

    traj_h = [h.detach().cpu()] if return_trajectory else None
    traj_xi = [] if return_xi_trajectory else None
    energies = []

    for _ in range(cfg.L):
        xi_input = h.detach() if cfg.causal_force else h
        xis = self.xi_module(xi_input)
        if return_xi_trajectory:
            traj_xi.append(xis.detach().cpu())

        h_in = h
        if not h_in.requires_grad:
            h_in = h_in.requires_grad_(True)

        V_out = self.V_theta(xis, h_in)       # (B, T, 1)
        V_scalar = V_out.sum()
        (grad_V,) = torch.autograd.grad(
            V_scalar, h_in,
            create_graph=self.training,
            retain_graph=True,
        )
        f = -grad_V
        v = (v + dt * f / m_b) / (1.0 + dt * gamma)
        h_new = h_in + dt * v
        if cfg.ln_after_step:
            h_new = self._project(h_new)
        h = h_new

        # Compute per-layer energy (mean over batch and tokens)
        v_norm2 = (v * v).sum(dim=-1)                       # (B, T)
        if isinstance(m_b, torch.Tensor) and m_b.dim() >= 2:
            m_scalar = m_b.squeeze(-1) if m_b.dim() > 2 else m_b
        else:
            m_scalar = m_b
        T_ell = (0.5 * m_scalar * v_norm2).mean()           # scalar
        V_ell = V_out.squeeze(-1).mean()                    # scalar
        energies.append((T_ell, V_ell))

        if return_trajectory:
            traj_h.append(h.detach().cpu())

    return h, traj_h, traj_xi, energies


def forward_with_shell_loss(
    self, x, targets=None,
    return_trajectory=False, return_xi_trajectory=False,
    shell_lambda=0.0,
):
    """Patched forward() that computes CE + soft energy-shell loss."""
    emb = self._embed(x)
    h_L, traj_h, traj_xi, energies = self.integrate_with_energy(
        x, emb,
        return_trajectory=return_trajectory,
        return_xi_trajectory=return_xi_trajectory,
    )
    logits = h_L @ self.E.weight.T

    ce_loss = None
    if targets is not None:
        ce_loss = F.cross_entropy(
            logits.reshape(-1, self.cfg.vocab_size),
            targets.reshape(-1),
        )

    # Soft energy-shell loss
    shell_loss = torch.tensor(0.0, device=x.device)
    if shell_lambda > 0 and len(energies) > 1:
        gamma_val = self.gamma.item() if hasattr(self.gamma, 'item') else float(self.gamma)
        E_layers = [T + V for T, V in energies]
        E0 = E_layers[0].detach()  # anchor: don't backprop through E0
        L = len(E_layers)
        for ell in range(L):
            E_target = E0 * math.exp(-gamma_val * (ell + 1))
            shell_loss = shell_loss + (E_layers[ell] - E_target) ** 2
        shell_loss = shell_lambda * shell_loss / L

    total_loss = None
    if ce_loss is not None:
        total_loss = ce_loss + shell_loss

    out = [logits, total_loss]
    # Attach diagnostics for logging
    self._last_ce_loss = ce_loss.item() if ce_loss is not None else None
    self._last_shell_loss = shell_loss.item()
    self._last_energies = [(T.item(), V.item()) for T, V in energies]

    if return_trajectory:
        out.append(traj_h)
    if return_xi_trajectory:
        out.append(traj_xi)
    return tuple(out) if len(out) > 2 else (out[0], out[1])


def patch_model(model):
    """Monkey-patch integrate and forward to support energy-shell loss."""
    model.integrate_with_energy = types.MethodType(integrate_with_energy, model)
    model._forward_with_shell = types.MethodType(forward_with_shell_loss, model)
    return model


print('Energy-shell patch defined.')

In [ ]:
# ── Cell 4: Training configuration ────────────────────────────────

TRAIN_STEPS = 2000
EVAL_INTERVAL = 200
EVAL_ITERS = 20
LOG_INTERVAL = 50
BLOCK_SIZE = 256
LR = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100

MODEL_D = 128
MODEL_L = 6
V_HIDDEN = 512
V_DEPTH = 3
XI_CHANNELS = 4

if DEVICE == 'cuda':
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE = 8 if vram_gb >= 30 else (4 if vram_gb >= 14 else 2)
else:
    BATCH_SIZE = 4

# Sweep configurations: (key, lambda, use_layernorm)
SWEEP = [
    ('baseline',      0.0,  True,  'tab:blue'),
    ('shell_0.01',    0.01, True,  'tab:cyan'),
    ('shell_0.1',     0.1,  True,  'tab:orange'),
    ('shell_1.0',     1.0,  True,  'tab:green'),
    ('shell_0.1_noLN', 0.1, False, 'tab:red'),
]

print(f'Training: {TRAIN_STEPS} steps, bs={BATCH_SIZE}, T={BLOCK_SIZE}')
print(f'Model: d={MODEL_D}, L={MODEL_L}')
print(f'Sweep: {len(SWEEP)} configurations')
for key, lam, ln, _ in SWEEP:
    print(f'  {key}: λ={lam}, LayerNorm={ln}')

In [ ]:
# ── Cell 5: Training loop ─────────────────────────────────────────

def lr_schedule(step, lr, warmup, total):
    if step < warmup:
        return lr * (step + 1) / warmup
    progress = (step - warmup) / max(total - warmup, 1)
    return lr * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def free_mem(model=None):
    if model is not None:
        model.cpu()
        del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()


@torch.no_grad()
def evaluate(model, ids, iters, batch_size, block_size, rng_eval, device):
    model.eval()
    losses = []
    for _ in range(iters):
        xb, yb = get_batch(ids, batch_size, block_size, rng_eval)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def train_one(key, shell_lambda, use_layernorm, seed=42):
    ckpt_path = DRIVE_CKPTS / f'{key}.pt'
    if ckpt_path.exists():
        print(f'\n  [{key}] Checkpoint exists, skipping.')
        return ckpt_path

    print(f'\n{"═" * 60}')
    print(f'Training: {key}  (λ={shell_lambda}, LN={use_layernorm})')
    print(f'{"═" * 60}')

    torch.manual_seed(seed)
    np.random.seed(seed)

    model_cfg = SPLMSARFMassLNMultiXiConfig(
        d=MODEL_D, max_len=1024,
        v_hidden=V_HIDDEN, v_depth=V_DEPTH, L=MODEL_L,
        init_m=1.0, init_gamma=1.0,
        vocab_size=50257,
        mass_mode='logfreq',
        logfreq_init_alpha=0.1,
        logfreq_path=LOGFREQ_PATH,
        ln_after_step=use_layernorm,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=[0.0, 0.5, 0.9, 0.99],
        xi_learnable=True,
    )
    model = ScalarPotentialLMSARFMassLNMultiXi(model_cfg)
    patch_model(model)
    model.to(DEVICE).train()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Parameters: {n_params:.2f}M')

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY,
    )
    rng_train = np.random.default_rng(seed)
    rng_eval = np.random.default_rng(seed + 1)

    train_losses, ce_losses, shell_losses = [], [], []
    energy_traces = []  # sample every LOG_INTERVAL
    val_ppls = []
    t0 = time.time()

    for step in range(TRAIN_STEPS):
        lr_now = lr_schedule(step, LR, WARMUP_STEPS, TRAIN_STEPS)
        for pg in optimizer.param_groups:
            pg['lr'] = lr_now

        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng_train)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)

        _, loss = model._forward_with_shell(x, y, shell_lambda=shell_lambda)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(loss.item())
        ce_losses.append(model._last_ce_loss)
        shell_losses.append(model._last_shell_loss)

        if step % LOG_INTERVAL == 0:
            elapsed = time.time() - t0
            ce_str = f'{model._last_ce_loss:.4f}' if model._last_ce_loss else 'N/A'
            print(f'  step {step:5d}/{TRAIN_STEPS}  '
                  f'total={loss.item():.4f}  ce={ce_str}  '
                  f'shell={model._last_shell_loss:.6f}  '
                  f'lr={lr_now:.2e}  {elapsed:.0f}s')
            energy_traces.append({
                'step': step,
                'energies': model._last_energies,
            })

        if (step + 1) % EVAL_INTERVAL == 0 or step == TRAIN_STEPS - 1:
            val_loss = evaluate(model, val_ids, EVAL_ITERS, BATCH_SIZE,
                                BLOCK_SIZE, rng_eval, DEVICE)
            val_ppl = math.exp(min(val_loss, 20.0))
            val_ppls.append((step, val_ppl))
            print(f'  [eval] step {step+1}  val_ppl={val_ppl:.2f}')

        del x, y, loss

    elapsed = time.time() - t0
    print(f'  Done in {elapsed:.0f}s ({elapsed/60:.1f} min)')

    model.cpu()
    torch.save({
        'config': asdict(model_cfg),
        'model_state_dict': model.state_dict(),
        'key': key,
        'shell_lambda': shell_lambda,
        'use_layernorm': use_layernorm,
        'train_losses': train_losses,
        'ce_losses': ce_losses,
        'shell_losses': shell_losses,
        'energy_traces': energy_traces,
        'val_ppls': val_ppls,
        'train_steps': TRAIN_STEPS,
        'elapsed_s': elapsed,
    }, ckpt_path)
    print(f'  Saved: {ckpt_path}')

    del optimizer
    free_mem(model)
    return ckpt_path


# ── Train all configurations ──
ckpt_paths = {}
for key, lam, ln, _ in SWEEP:
    ckpt_paths[key] = train_one(key, lam, ln)

print(f'\nAll {len(SWEEP)} training runs complete.')

In [ ]:
# ── Cell 6: Training curves ───────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for key, _, _, color in SWEEP:
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)

    # Smoothed CE loss
    ce = np.array([c for c in ckpt['ce_losses'] if c is not None])
    w = min(50, len(ce) // 4)
    if w > 1:
        ce_s = np.convolve(ce, np.ones(w)/w, mode='valid')
        axes[0, 0].plot(range(w-1, len(ce)), ce_s, label=key, color=color, alpha=0.8)
    else:
        axes[0, 0].plot(ce, label=key, color=color, alpha=0.8)

    # Shell loss
    sl = np.array(ckpt['shell_losses'])
    if w > 1 and sl.max() > 0:
        sl_s = np.convolve(sl, np.ones(w)/w, mode='valid')
        axes[0, 1].plot(range(w-1, len(sl)), sl_s, label=key, color=color, alpha=0.8)

    # Val PPL
    vp = ckpt['val_ppls']
    axes[1, 0].plot([v[0] for v in vp], [v[1] for v in vp], 'o-',
                    label=key, color=color, markersize=4)

    # Energy traces (last recorded)
    if ckpt['energy_traces']:
        last_e = ckpt['energy_traces'][-1]['energies']
        E_total = [T + V for T, V in last_e]
        axes[1, 1].plot(range(1, len(E_total) + 1), E_total, 'o-',
                        label=key, color=color, markersize=4)

axes[0, 0].set_title('CE Loss (smoothed)'); axes[0, 0].set_ylabel('CE Loss')
axes[0, 1].set_title('Shell Loss (smoothed)'); axes[0, 1].set_ylabel('Shell Loss')
axes[1, 0].set_title('Validation Perplexity'); axes[1, 0].set_ylabel('PPL')
axes[1, 1].set_title('Final Energy Profile E(ℓ)'); axes[1, 1].set_ylabel('E = T + V')
axes[1, 1].set_xlabel('Layer')

for ax in axes.flat:
    ax.set_xlabel('Step') if ax != axes[1, 1] else None
    ax.legend(fontsize=7)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Soft Energy-Shell Regularisation: Training Dynamics', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(DRIVE_RESULTS / 'training_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {DRIVE_RESULTS / "training_curves.png"}')

In [ ]:
# ── Cell 7: Arm 2 Diagnostic — Geodesic Compliance ────────────────

N_EVAL_BATCHES = 3
EVAL_BATCH_SIZE = 4
EVAL_BLOCK_SIZE = 128

rng_diag = np.random.default_rng(123)
eval_batches = []
for _ in range(N_EVAL_BATCHES):
    xb, _ = get_batch(val_ids, EVAL_BATCH_SIZE, EVAL_BLOCK_SIZE, rng_diag)
    eval_batches.append(torch.tensor(xb))


def load_sweep_model(key, use_layernorm):
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    cfg_dict = ckpt['config']
    known = {f.name for f in dc_fields(SPLMSARFMassLNMultiXiConfig)}
    model_cfg = SPLMSARFMassLNMultiXiConfig(
        **{k: v for k, v in cfg_dict.items() if k in known}
    )
    if hasattr(model_cfg, 'logfreq_path'):
        model_cfg.logfreq_path = LOGFREQ_PATH
    model = ScalarPotentialLMSARFMassLNMultiXi(model_cfg)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    model.to(DEVICE).eval()
    n = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Loaded {key}: {n:.2f}M params')
    return model


def extract_trajectory(model, x):
    with torch.enable_grad():
        out = model(x, targets=None, return_trajectory=True,
                    return_xi_trajectory=False)
        logits, _loss, traj = out[0], out[1], out[2]
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    traj_cpu = [h.detach().cpu() for h in traj]
    logits_cpu = logits.detach().cpu()
    del traj, out, logits
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return traj_cpu, logits_cpu


def compute_grad_V(model, h):
    h_in = h.detach().requires_grad_(True)
    xis = model.xi_module(h_in.detach())
    V = model.V_theta(xis, h_in)
    grad_V = torch.autograd.grad(V.sum(), h_in, create_graph=False)[0]
    return grad_V.detach()


def get_mass(model, x):
    emb = model._embed(x)
    m = model.compute_mass(x, emb)
    return m.detach().cpu() if isinstance(m, torch.Tensor) else m


print('═' * 60)
print('ARM 2: Geodesic Compliance (all sweep configurations)')
print('═' * 60)

arm2_all = {}

for key, lam, ln, color in SWEEP:
    print(f'\n── {key} (λ={lam}, LN={ln}) ──')
    model = load_sweep_model(key, ln)

    gamma = model.gamma.item() if hasattr(model.gamma, 'item') else float(model.gamma)
    print(f'  γ = {gamma:.6f}')

    comp_und, comp_dmp = None, None
    cos_und, cos_dmp = None, None

    for bi in range(N_EVAL_BATCHES):
        x = eval_batches[bi].to(DEVICE)
        traj, _ = extract_trajectory(model, x)
        L = len(traj) - 1

        if comp_und is None:
            comp_und = [[] for _ in range(L - 1)]
            comp_dmp = [[] for _ in range(L - 1)]
            cos_und  = [[] for _ in range(L - 1)]
            cos_dmp  = [[] for _ in range(L - 1)]

        with torch.no_grad():
            m = get_mass(model, x)
            m_flat = m.squeeze(-1) if isinstance(m, torch.Tensor) and m.dim() > 2 else m
        del x

        for ell in range(1, L):
            h_prev = traj[ell - 1].to(DEVICE)
            h_curr = traj[ell].to(DEVICE)
            h_next = traj[ell + 1].to(DEVICE)

            v = h_curr - h_prev
            a_obs = h_next - 2 * h_curr + h_prev
            del h_prev, h_next

            grad_V = compute_grad_V(model, h_curr)

            v_norm2 = (v ** 2).sum(dim=-1, keepdim=True)
            mf = (m_flat.unsqueeze(-1).to(DEVICE)
                  if isinstance(m_flat, torch.Tensor) and m_flat.dim() == 2
                  else m_flat)
            ke = 0.5 * mf * v_norm2
            denom = (2.0 * ke).clamp(min=1e-8)

            gV_dot_v = (grad_V * v).sum(dim=-1, keepdim=True)
            a_jacobi = (2.0 * v * gV_dot_v - v_norm2 * grad_V) / denom
            a_damped = a_jacobi - gamma * v

            a2 = (a_obs ** 2).sum(dim=-1).mean().item()
            r2_u = ((a_obs - a_jacobi) ** 2).sum(dim=-1).mean().item()
            r2_d = ((a_obs - a_damped) ** 2).sum(dim=-1).mean().item()
            comp_und[ell-1].append(1.0 - r2_u / max(a2, 1e-12))
            comp_dmp[ell-1].append(1.0 - r2_d / max(a2, 1e-12))

            af = a_obs.reshape(-1, a_obs.shape[-1])
            cos_und[ell-1].append(F.cosine_similarity(
                af, a_jacobi.reshape(-1, a_jacobi.shape[-1]), dim=-1).mean().item())
            cos_dmp[ell-1].append(F.cosine_similarity(
                af, a_damped.reshape(-1, a_damped.shape[-1]), dim=-1).mean().item())

            del h_curr, v, a_obs, grad_V, a_jacobi, a_damped

        del traj
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    arm2_all[key] = {
        'r2_und': [np.mean(l) for l in comp_und],
        'r2_dmp': [np.mean(l) for l in comp_dmp],
        'cos_und': [np.mean(l) for l in cos_und],
        'cos_dmp': [np.mean(l) for l in cos_dmp],
        'r2_und_mean': np.mean([np.mean(l) for l in comp_und]),
        'r2_dmp_mean': np.mean([np.mean(l) for l in comp_dmp]),
        'cos_und_mean': np.mean([np.mean(l) for l in cos_und]),
        'cos_dmp_mean': np.mean([np.mean(l) for l in cos_dmp]),
    }
    a2 = arm2_all[key]
    print(f'  R²(dmp)={a2["r2_dmp_mean"]:.4f}  cos(dmp)={a2["cos_dmp_mean"]:.4f}  '
          f'R²(und)={a2["r2_und_mean"]:.4f}  cos(und)={a2["cos_und_mean"]:.4f}')

    free_mem(model); del model

print('\nArm 2 complete. ✓')

In [ ]:
# ── Cell 8: Results plots + summary ───────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for key, _, _, color in SWEEP:
    a2 = arm2_all[key]
    layers = range(1, len(a2['cos_dmp']) + 1)
    axes[0, 0].plot(layers, a2['cos_und'], 'o--', label=key, color=color, markersize=3, alpha=0.7)
    axes[0, 1].plot(layers, a2['cos_dmp'], 'o-',  label=key, color=color, markersize=3)
    axes[1, 0].plot(layers, a2['r2_und'],  's--', label=key, color=color, markersize=3, alpha=0.7)
    axes[1, 1].plot(layers, a2['r2_dmp'],  's-',  label=key, color=color, markersize=3)

axes[0, 0].set_title('Cosine (Undamped)'); axes[0, 0].set_ylabel('Cosine Sim')
axes[0, 1].set_title('Cosine (Damped)'); axes[0, 1].set_ylabel('Cosine Sim')
axes[1, 0].set_title('R² Magnitude (Undamped)'); axes[1, 0].set_ylabel('R²')
axes[1, 1].set_title('R² Magnitude (Damped)'); axes[1, 1].set_ylabel('R²')
for ax in axes.flat:
    ax.set_xlabel('Layer'); ax.legend(fontsize=7)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
axes[1, 0].axhline(0, color='gray', ls=':', lw=0.8)
axes[1, 1].axhline(0, color='gray', ls=':', lw=0.8)

plt.suptitle('Soft Energy-Shell: Arm 2 Geodesic Compliance', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(DRIVE_RESULTS / 'arm2_shell.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()


# ── Summary bar chart ──
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))
keys = [k for k, _, _, _ in SWEEP]
colors = [c for _, _, _, c in SWEEP]
cos_vals = [arm2_all[k]['cos_dmp_mean'] for k in keys]
r2_vals  = [arm2_all[k]['r2_dmp_mean'] for k in keys]

x_pos = range(len(keys))
axes2[0].bar(x_pos, cos_vals, color=colors)
axes2[0].set_xticks(x_pos); axes2[0].set_xticklabels(keys, rotation=30, ha='right', fontsize=8)
axes2[0].set_ylabel('Cosine (damped)'); axes2[0].set_title('Directional Compliance')
axes2[1].bar(x_pos, r2_vals, color=colors)
axes2[1].set_xticks(x_pos); axes2[1].set_xticklabels(keys, rotation=30, ha='right', fontsize=8)
axes2[1].set_ylabel('R² (damped)'); axes2[1].set_title('Magnitude Compliance')
axes2[1].axhline(0, color='gray', ls=':', lw=0.8)
for ax in axes2:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Soft Energy-Shell: Summary', fontweight='bold', y=1.02)
plt.tight_layout()
fig2.savefig(DRIVE_RESULTS / 'arm2_shell_summary.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved plots to {DRIVE_RESULTS}')

In [ ]:
# ── Cell 9: Save full results JSON ─────────────────────────────────

final_ppls = {}
for key, _, _, _ in SWEEP:
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    vp = ckpt['val_ppls']
    final_ppls[key] = vp[-1][1] if vp else float('nan')

results = {
    'config': {
        'model_d': MODEL_D, 'model_L': MODEL_L,
        'v_hidden': V_HIDDEN, 'v_depth': V_DEPTH,
        'train_steps': TRAIN_STEPS, 'batch_size': BATCH_SIZE,
        'block_size': BLOCK_SIZE,
    },
    'sweep': {},
}

for key, lam, ln, _ in SWEEP:
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    results['sweep'][key] = {
        'shell_lambda': lam,
        'use_layernorm': ln,
        'final_val_ppl': final_ppls[key],
        'final_ce_loss': ckpt['ce_losses'][-1] if ckpt['ce_losses'] else None,
        'final_shell_loss': ckpt['shell_losses'][-1] if ckpt['shell_losses'] else None,
        'energy_profile_final': ckpt['energy_traces'][-1]['energies'] if ckpt['energy_traces'] else None,
        'arm2': arm2_all.get(key, {}),
    }

report_path = DRIVE_RESULTS / 'soft_energy_shell_report.json'
with open(report_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'Full results: {report_path}')


print('\n' + '═' * 90)
print('SOFT ENERGY-SHELL REGULARISATION: SUMMARY')
print('═' * 90)
print(f'{"Config":<20} {"λ":>6} {"LN":>5} {"PPL":>8} {"Cos(dmp)":>10} '
      f'{"R²(dmp)":>10} {"Shell loss":>12}')
print('-' * 90)
for key, lam, ln, _ in SWEEP:
    a2 = arm2_all.get(key, {})
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    sl = ckpt['shell_losses'][-1] if ckpt['shell_losses'] else 0
    print(f'{key:<20} {lam:>6.2f} {str(ln):>5} {final_ppls[key]:>8.1f} '
          f'{a2.get("cos_dmp_mean", float("nan")):>10.4f} '
          f'{a2.get("r2_dmp_mean", float("nan")):>10.4f} '
          f'{sl:>12.6f}')
print('═' * 90)
print('\n✓ Experiment complete.')